In [ ]:
import asyncio
from crawl4ai import *
import httpx
import time
import json

from bs4 import BeautifulSoup
import re

In [ ]:
text = ""
result = None
async with AsyncWebCrawler() as crawler:
    result = await crawler.arun(
        url="https://spesaonline.conad.it/c/formaggi--0401",
    )
    text=result.html
    
text


In [ ]:
html = result.html

soup = BeautifulSoup(html, "html.parser")

divs = soup.select('div[data-product]')

results = []

for div in divs:
    raw = div.get("data-product")
    if not raw:
        continue

    product = json.loads(raw)

    # pull the badge text from the rendered HTML (not in data-product)
    badge_el = div.select_one(".badge-territorio .text")
    badge = badge_el.get_text(strip=True) if badge_el else None

    results.append({
        "code": product.get("code"),
        "name": product.get("nome"),
        "brand": product.get("marchio"),
        "category_l1": product.get("categoriaPrimoLivello"),
        "category_l2": product.get("categoriaSecondoLivello"),
        "category_l3": product.get("categoriaTerzoLivello"),
        "base_price": product.get("basePrice"),
        "unit_of_measure": product.get("increment", {}).get("unitOfMeasure"),
        "min_weight": product.get("increment", {}).get("minWeight"),
        "net_quantity_um": product.get("netQuantityUm"),
        "marketing_badge": product.get("marketingCategoryBadge"),
        "badge_label": badge,
        "image_url": product.get("defaultImgSrc"),
        "product_url": f"https://spesaonline.conad.it/p/{div.select_one('a.product')['href'].lstrip('/') if div.select_one('a.product') else ''}",
    })

for r in results:
    print(json.dumps(r, indent=2, ensure_ascii=False))
    
#json dump
open("scrape_conad_products.json", "w", encoding="utf-8").write(json.dumps(results, indent=2, ensure_ascii=False))

In [ ]:
product = results[0]
url=product["product_url"]
url = url.replace("/p/p/", "/p/", 1)
print(url)

product_text=""
async with AsyncWebCrawler() as crawler:
    result = await crawler.arun(
        url=url,
    )
    product_text=result.markdown
    result = result

In [ ]:
ean = None
def parse_product_page(html):
    soup = BeautifulSoup(html, "html.parser")
    product_info = soup.find("div", {"class": "product-other-info"})


    main = soup.find('main', attrs={'data-product': True})
    product_data = json.loads(main['data-product'])
    ean = product_data['ean']
    print(f"Extracted EAN: {ean}, this can be used as an identifier for the product.")

    if not product_info:
        return None
    text = product_info.get_text(" ", strip=True)
    return (text, ean)


In [ ]:
import json
from groq import Groq
import re
import time
import os
from dotenv import load_dotenv

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

def extract_ucpd_claims_groq(product, description, max_retries=3):
    schema = {
        "claims": [
            {
                "claim_text": "exact text as found in description",
                "category": "HEALTH_CLAIM | ORIGIN_CLAIM | NATURALNESS_CLAIM | FREE_FROM_CLAIM | ENVIRONMENTAL_CLAIM | COMPARATIVE_CLAIM | QUALITY_CLAIM | ABSOLUTE_CLAIM",
                "risk_level": "HIGH | MEDIUM | LOW",
                "risk_rationale": "specific reason this risk level applies",
                "ucpd_article": "most relevant UCPD article (e.g. Art.6 misleading actions)",
                "regulation": "primary applicable EU regulation"
            }
        ],
        "badge_claims": [
            {
                "claim_text": "exact badge or label text",
                "badge_source": "name of certification body or standard if identifiable, else null",
                "verifiable": "true if backed by a known EU certification scheme, false otherwise",
                "category": "same categories as above",
                "risk_level": "HIGH | MEDIUM | LOW",
                "risk_rationale": "specific reason this risk level applies",
                "regulation": "primary applicable EU regulation"
            }
        ]
    }

    system_prompt = f"""You are a regulatory analyst specializing in EU consumer protection law.
Your task is to identify and classify marketing claims that may require regulatory verification.

APPLICABLE REGULATIONS:
- UCPD 2005/29/EC (Unfair Commercial Practices Directive): prohibits misleading actions (Art.6), misleading omissions (Art.7), and aggressive practices (Art.8-9)
- EU Health Claims Regulation 1924/2006: all nutrition and health claims must be pre-authorized
- EU Green Claims Directive 2024/825/EU: environmental claims must be substantiated and verified
- EU Food Information Regulation 1169/2011: origin and composition claims
- PDO/PGI/TSG Regulation 1151/2012: protected designations of origin

RISK LEVEL CRITERIA:
- HIGH: claim is likely unlawful without substantiation (unverified health claims, false origins, absolute environmental claims like "eco-friendly" or "sustainable" without certification, superlatives like "best" or "purest")
- MEDIUM: claim requires evidence that may exist but is not shown (e.g. "natural", "traditional", "artisanal", "high quality", vague origin claims)  
- LOW: claim is factual and verifiable from product specs, or is a recognized/certified label (e.g. DOP, IGP, EU Organic)

CLAIM CATEGORIES:
- HEALTH_CLAIM: references to health benefits, nutrients, body functions
- ORIGIN_CLAIM: geographic, regional, national origin claims
- NATURALNESS_CLAIM: natural, pure, traditional, artisanal, genuine
- FREE_FROM_CLAIM: absence of ingredients (gluten-free, no preservatives, etc.)
- ENVIRONMENTAL_CLAIM: sustainability, eco, green, biodegradable, carbon-neutral
- COMPARATIVE_CLAIM: better than, superior to, number one
- QUALITY_CLAIM: premium, excellence, superior quality, best
- ABSOLUTE_CLAIM: always, never, 100%, only, unique — any absolute statement

OUTPUT: Return only a valid JSON object matching this exact schema:
{json.dumps(schema, indent=2, ensure_ascii=False)}

If no claims are found in a category, return an empty array.
Do not invent claims. Only extract text that is explicitly present in the input."""

    user_prompt = f"""Analyze the following Italian food product and extract all claims requiring regulatory verification.

PRODUCT NAME: {product.get("name", "N/A")}
BRAND: {product.get("brand", "N/A")}
CATEGORY: {product.get("category_l3", product.get("category_l2", "N/A"))}
MARKETING BADGE (assigned by retailer): {product.get("marketing_badge", "none")}
BADGE LABEL (displayed on product): {product.get("badge_label", "none")}
PRIVATE LABEL LINE: {product.get('brand', 'N/A')} — treat the line name as a potential implicit claim if it suggests health, wellness, naturalness, or sustainability

PRODUCT DESCRIPTION:
{description if description else "No description available"}

Extract claims from both the description and the badge/label fields. Treat the badge and badge_label fields as badge_claims."""



    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(model="llama-3.3-70b-versatile",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                response_format={"type": "json_object"},
                temperature=0.1,
                max_tokens=2048)
            return json.loads(response.choices[0].message.content)
        except json.JSONDecodeError as e:
            print(f"  Bad JSON attempt {attempt+1} for {product['ean']}: {e}")
        except Exception as e:
            print(f"  Error attempt {attempt+1} for {product['ean']}: {e}")
            if "rate_limit" in str(e).lower():
                # extract wait time from error message e.g. "try again in 18.22s"
                match = re.search(r"try again in ([\d.]+)s", str(e))
                wait = float(match.group(1)) + 1 if match else 30
                print(f"  Rate limited, waiting {wait}s...")
                time.sleep(wait)
            else:
                break 

    return None

In [ ]:
claims =extract_ucpd_claims(product, text)

In [ ]:
import time
from crawl4ai.async_dispatcher import MemoryAdaptiveDispatcher, RateLimiter

browser_config = BrowserConfig(
    headless=True,
    # use_managed_browser=True,
    use_managed_browser=True,
    extra_args=["--disable-blink-features=AutomationControlled"],
    headers={
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
        "Accept-Language": "it-IT,it;q=0.9",
        "Accept": "text/html,application/xhtml+xml,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    }
)



dispatcher = MemoryAdaptiveDispatcher(
    max_session_permit=1,  # how many concurrent requests, lower = gentler
    rate_limiter=RateLimiter(
        base_delay=(2.0, 5.0),  # random delay range between requests, seconds
        max_delay=30.0,
        max_retries=3,
        rate_limit_codes=[301, 429, 503]  # treat these as "back off and retry"
    )
)
product_results = json.loads(open("results/scrape_conad_products.json", "r", encoding="utf-8").read())
    
scrape_results = []
urls = [
    product["product_url"].replace("/p/p/", "/p/", 1)
    for product in product_results
    if product.get("product_url")
]



In [ ]:
run_config_homepage = CrawlerRunConfig(
    js_code="""
        
       // accept cookies first
    await new Promise((resolve, reject) => {
        const interval = setInterval(() => {
            const cookieBtn = document.getElementById('onetrust-accept-btn-handler');
            if (cookieBtn) {
                clearInterval(interval);
                cookieBtn.click();
                resolve();
            }
        }, 300);
        setTimeout(() => { clearInterval(interval); reject(new Error('cookie banner never appeared')); }, 10000);
    });
    
    await new Promise(r => setTimeout(r, 4000));
    
    // then do the address stuff
    const input = document.getElementById('googleInputEntrypageLine1');
    """,
    page_timeout=30000,
    log_console=True,
    capture_console_messages=True,
)

import asyncio, json
browser_config = BrowserConfig(
    headless=False,
    # use_managed_browser=True,
    use_managed_browser=True,
    extra_args=["--disable-blink-features=AutomationControlled"],
    headers={
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
        "Accept-Language": "it-IT,it;q=0.9",
        "Accept": "text/html,application/xhtml+xml,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    }
)


In [ ]:
parsed_products = []
failed_products = []
shortened_urls = urls  # limit to first 5 for testing

async with AsyncWebCrawler(config=browser_config) as crawler:
    result = await crawler.arun(url="https://spesaonline.conad.it", config=run_config_homepage)
    if result.console_messages:
        for msg in result.console_messages:
            print(msg)
    await asyncio.sleep(3)
    
    for url in shortened_urls:
        result = await crawler.arun(url=url)
        try:
            parsed, ean = parse_product_page(result.html)
        except Exception as e:
            failed_products.append({"url": result.url, "reason": f"parse_error: {e}"})
            await asyncio.sleep(5)
            continue

        if not ean:
            failed_products.append({"url": result.url, "reason": "no_ean_found"})
            await asyncio.sleep(5)
            continue

        parsed_products.append({
            "ean": ean,
            "url": result.url,
            "description": parsed
        })

        with open("results/parsed_products.json", "w", encoding="utf-8") as f:
            json.dump(parsed_products, f, ensure_ascii=False, indent=2)

        print(f"ok ({len(parsed_products)}) {result.url}")
        await asyncio.sleep(5)

In [ ]:
# save parsed products with ean to json
with open("raw_text/06.26.json", "w", encoding="utf-8") as f:
    json.dump(parsed_products, f, indent=2, ensure_ascii=False)


In [ ]:
import json
from concurrent.futures import ThreadPoolExecutor, as_completed

parsed_products = json.loads(open("parsed_conad_products.json", "r", encoding="utf-8").read())

def process_product(product):
    print(f"Extracting claims for product {product['ean']}...")
    product["claims"] = extract_ucpd_claims(product, product["description"])
    return product

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(process_product, p): p for p in parsed_products}
    results = []
    for future in as_completed(futures):
        results.append(future.result())

# as_completed doesn't preserve order, so re-sort by original index
original_order = {p["ean"]: i for i, p in enumerate(parsed_products)}
results.sort(key=lambda p: original_order[p["ean"]])

with open("parsed_conad_products_with_claims.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

Example of parsing with local Ollama llm which is single threaded, hence slower.


In [ ]:
import json

parsed_products = json.loads(open("parsed_conad_products.json", "r", encoding="utf-8").read())
output_file = "conad_products_with_claims.json"

try:
    existing = json.loads(open(output_file, "r", encoding="utf-8").read())
    done_eans = {p["ean"] for p in existing if "claims" in p}
    results = existing
    print(f"Resuming: {len(done_eans)} already done")
except FileNotFoundError:
    done_eans = set()
    results = []

todo = [p for p in parsed_products if p["ean"] not in done_eans]
print(f"Products left: {len(todo)}")

for product in todo:
    print(f"Extracting claims for {product['ean']}...")
    product["claims"] = extract_ucpd_claims(product, product["description"])
    results.append(product)
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

print("Done!")

Trying with GROQ which is faster and more reliable, but it's rate limited for the free tier.

In [ ]:
import json
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

parsed_products = json.loads(open("parsed_conad_products.json", "r", encoding="utf-8").read())
output_file = "groq_conad_products_with_claims.json"

try:
    existing = json.loads(open(output_file, "r", encoding="utf-8").read())
    done_eans = {p["ean"] for p in existing if "claims" in p}
    results = existing
    print(f"Resuming: {len(done_eans)} already done")
except FileNotFoundError:
    done_eans = set()
    results = []


def process_product(product):
    print(f"Extracting claims for {product['ean']}...")
    product["claims"] = extract_ucpd_claims_groq(product, product["description"])
    return product

todo = [p for p in parsed_products if p["ean"] not in done_eans]
print(f"Products left: {len(todo)}")

with ThreadPoolExecutor(max_workers=3) as executor:
    futures = {executor.submit(process_product, p): p for p in todo}
    for future in as_completed(futures):
        product = future.result()
        results.append(product)
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)

original_order = {p["ean"]: i for i, p in enumerate(parsed_products)}
results.sort(key=lambda p: original_order[p["ean"]])
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Done!")